# 20.07 - Frame Aggregation Baseline

**Notebook type:** Practice notebook with theory, exercises, TODO cells, smoke checks, and test cases.

**Daily output:** A CNN-style frame classifier applied to sampled video frames, with video-level logits produced by middle-frame, uniform-frame mean, and max aggregation.

Today builds the simplest useful video-classification baseline: reuse an image model on each sampled frame, restore the temporal dimension, and combine frame evidence into one prediction per video. The notebook uses a deterministic color-based frame model so the aggregation behavior is visible without downloads or a long training run.

## Core Ideas

### 1. Keep video and class axes separate

A sampled video batch has shape `[B, T, C, H, W]`. An image model expects `[N, C, H, W]`, so flatten only the first two axes to `[B*T, C, H, W]`. After inference, reshape `[B*T, K]` frame logits back to `[B, T, K]`, where `K` is the number of classes. Mixing the batch and time axes silently combines evidence from different videos.

### 2. Three practical baselines

- **Middle frame:** classify only one representative frame. It is cheap, but it can miss an action or land on a misleading moment.
- **Mean logits:** average each class score over uniformly sampled frames. It rewards evidence that is consistent over time and is the primary baseline for today.
- **Max logits:** keep the strongest score observed for each class. It can capture a brief event, but one noisy frame can dominate.

### 3. Aggregate logits consistently

This notebook aggregates raw logits, as required by the masterplan. Averaging logits, averaging probabilities, and voting over hard labels are different rules and can yield different predictions. Record the rule with every experiment.

### 4. Inference discipline and speed

Use `model.eval()` and `torch.no_grad()` for deterministic, memory-efficient inference. Middle-frame inference processes `B` images; mean and max normally process `B*T`. Vectorizing frames is simple and fast, but chunking may be needed when `B*T` is too large for memory.

In [2]:
import torch
import torch.nn as nn

SEED = 20
torch.manual_seed(SEED)


## Prepared Video Batch

The provided cell creates four five-frame RGB videos with labels `0 = red_event` and `1 = blue_event`. Most frames contain the correct class color, while the middle frame is a stronger opposite-color distractor. This makes mean aggregation correct and exposes the failure modes of middle and max aggregation. Data preparation is provided and is not a learner TODO.

- `video_batch`: CPU `torch.float32` tensor `[B, T, C, H, W] = [4, 5, 3, 8, 8]`, with RGB values in `[0, 1]`.
- `video_labels`: CPU `torch.long` tensor `[B] = [4]`, with values in `{0, 1}`.
- `class_names`: `list[str]` of length 2 in label-ID order.

In [3]:
B, T, C, H, W = 4, 5, 3, 8, 8
video_labels = torch.tensor([0, 1, 0, 1], dtype=torch.long)
class_names = ["red_event", "blue_event"]
video_batch = torch.full((B, T, C, H, W), 0.05, dtype=torch.float32)

for video_index, label in enumerate(video_labels.tolist()):
    correct_channel = 0 if label == 0 else 2
    distractor_channel = 2 if label == 0 else 0
    video_batch[video_index, :, correct_channel] = 0.75
    video_batch[video_index, :, distractor_channel] = 0.10
    video_batch[video_index, T // 2, correct_channel] = 0.20
    video_batch[video_index, T // 2, distractor_channel] = 0.90

assert video_batch.shape == (4, 5, 3, 8, 8)
assert video_batch.dtype == torch.float32
print("Prepared video batch:", tuple(video_batch.shape))


Prepared video batch: (4, 5, 3, 8, 8)


## Prepared Image Model

`MeanColorFrameClassifier` is a provided deterministic image model. It turns the spatial mean of the red and blue channels into two class logits. It has no trainable parameters; its purpose is to isolate and explain temporal aggregation.

**Return structure — `MeanColorFrameClassifier()`:** one `torch.nn.Module` on the CPU with no trainable parameters. Calling it with a CPU `torch.float32` RGB tensor `frames` of shape `[N, 3, H, W]` returns a CPU `torch.float32` tensor of logits with shape `[N, K]`, where `K = 2`; column 0 is the mean red score and column 1 is the mean blue score. Invalid input shape raises `ValueError`.

In [4]:
class MeanColorFrameClassifier(nn.Module):
    def forward(self, frames):
        if frames.ndim != 4 or frames.shape[1] != 3:
            raise ValueError("frames must have shape [N, 3, H, W]")
        mean_rgb = frames.mean(dim=(2, 3))
        return torch.stack((mean_rgb[:, 0], mean_rgb[:, 2]), dim=1)

frame_model = MeanColorFrameClassifier()
with torch.no_grad():
    prepared_frame_logits = frame_model(
        video_batch.reshape(B * T, C, H, W)
    ).reshape(B, T, 2)
print("Prepared frame logits:", tuple(prepared_frame_logits.shape))


Prepared frame logits: (4, 5, 2)


## Exercise 20-A: Select Middle-Frame Logits

Implement `middle_frame_logits(frame_logits)`. For an odd number of frames, select the exact middle. For an even number, use the upper-middle index `T // 2`. Keep the batch and class axes unchanged.

**Return structure — `middle_frame_logits(frame_logits)`:** one `torch.Tensor` with shape `[B, K]`, with the same floating dtype and device as the input `frame_logits`. The input must have shape `[B, T, K]` with `B > 0`, `T > 0`, and `K > 0`; invalid shape raises `ValueError`.

In [5]:
# TODO 20-A
def middle_frame_logits(frame_logits):
    # Validate [B, T, K], compute T // 2, and select that time step.
    if frame_logits.ndim != 3 or min(frame_logits.shape) <= 0 : 
        raise ValueError()
    return frame_logits[:,frame_logits.shape[1]//2,:]


# Smoke check: inspect one video-level logit row.
middle_smoke = middle_frame_logits(prepared_frame_logits)
print("Middle-frame smoke check:", middle_smoke[0].tolist())


Middle-frame smoke check: [0.20000001788139343, 0.9000000357627869]


## Exercise 20-B: Aggregate Frame Logits

Implement `aggregate_frame_logits(frame_logits, method="mean")`. Support exactly `"middle"`, `"mean"`, and `"max"`. Reduce only the temporal axis; never reduce over videos or classes.

**Return structure — `aggregate_frame_logits(frame_logits, method)`:** one `torch.Tensor` with shape `[B, K]`, with the same floating dtype and device as the input `[B, T, K]` tensor. `method` must be one of `"middle"`, `"mean"`, or `"max"`; invalid shapes raise `ValueError` and unsupported methods raise `ValueError`.

In [12]:
# TODO 20-B
def aggregate_frame_logits(frame_logits, method="mean"):
    if frame_logits.ndim != 3 or min(frame_logits.shape) <= 0 : 
        raise ValueError()
    if method == "mean" :
        return frame_logits.mean(dim = 1)
    if method == "middle" :
        return middle_frame_logits(frame_logits)
    if method == "max" :
        return frame_logits.max(dim = 1)[0]
    raise ValueError()
    # Validate [B, T, K], then reduce only dimension 1.
    # Reuse middle_frame_logits for the middle strategy.

# Smoke check: mean aggregation should return one row per video.
mean_smoke = aggregate_frame_logits(prepared_frame_logits, method="mean")
print("Mean-logit smoke check:", tuple(mean_smoke.shape), mean_smoke[0].tolist())


Mean-logit smoke check: (4, 2) [0.6399999856948853, 0.2600000202655792]


## Exercise 20-C: Run an Image Model Across a Video Batch

Implement `video_frame_logits(model, video_batch)`. Flatten `[B, T, C, H, W]` into `[B*T, C, H, W]`, run the image model once in evaluation mode under `torch.no_grad()`, validate its output, and restore `[B, T, K]`.

**Return structure — `video_frame_logits(model, video_batch)`:** one `torch.Tensor` with shape `[B, T, K]`, on the same device as `video_batch`, where `K > 0` is inferred from the model output. Its floating dtype is the model output dtype. The input must be a non-empty tensor `[B, T, C, H, W]`; the model must return a tensor `[B*T, K]`. Invalid contracts raise `TypeError` or `ValueError`. The function also leaves `model.training == False`.

In [7]:
# TODO 20-C
def video_frame_logits(model, video_batch):
    if not isinstance(model, nn.Module) : 
        raise TypeError()
    if not isinstance(video_batch, torch.Tensor) : 
        raise TypeError()
    if video_batch.ndim != 5 or min(video_batch.shape) <= 0 :
        raise ValueError()
    B, T, C, H, W = video_batch.shape
    flat = torch.flatten(video_batch, start_dim = 0, end_dim = 1)
    device = video_batch.device
    model.to(device)
    model.eval()
    with torch.no_grad() : 
        logits = model(flat)
        if not isinstance(logits, torch.Tensor) : 
            raise TypeError()
        if logits.ndim != 2 : 
            raise ValueError()
        if logits.shape[0] != B * T : 
            raise ValueError()
    return torch.reshape(logits,(B,T,-1))


# Smoke check: recover one logit vector per frame and per video.
frame_logits_smoke = video_frame_logits(frame_model, video_batch)
print("Frame-inference smoke check:", tuple(frame_logits_smoke.shape))


Frame-inference smoke check: (4, 5, 2)


## Exercise 20-D: Compare Video-Level Aggregation Strategies

Implement `evaluate_aggregation_methods(model, video_batch, labels)`. Run frame inference once, evaluate the three aggregation methods, and return predictions plus accuracy for each. Reusing frame logits keeps this comparison focused on aggregation rather than repeated model work.

**Return structure — `evaluate_aggregation_methods(model, video_batch, labels)`:** a `dict[str, dict]` with exactly three outer keys: `"middle"`, `"mean"`, and `"max"`. Each nested dictionary has exactly: `"logits"`, a CPU or device-matched floating `torch.Tensor` `[B, K]`; `"predictions"`, a `torch.long` tensor `[B]` on the same device; and `"accuracy"`, a Python `float` in `[0.0, 1.0]`. `labels` must be a `torch.long` tensor `[B]` on the same device as the batch and contain class IDs in `[0, K-1]`; invalid contracts raise `TypeError` or `ValueError`.

In [14]:
# TODO 20-D
def evaluate_aggregation_methods(model, video_batch, labels):
    # Validate labels, compute frame logits once, and build the exact
    # nested report schema for middle, mean, and max aggregation.
    if labels.device != video_batch.device :
        raise TypeError()
    if not isinstance(labels, torch.Tensor) :
        raise TypeError()
    if labels.dtype != torch.long or labels.ndim != 1 :
        raise ValueError() 
    if labels.shape[0] != video_batch.shape[0] :
        raise ValueError()

    frame_logits = video_frame_logits(model, video_batch)
    report = {}
    for method in ("middle", "mean", "max"):
        logits = aggregate_frame_logits(frame_logits, method=method)
        predictions = logits.argmax(dim=1)
        accuracy = float((predictions == labels).float().mean().item())
        report[method] = {
            "logits": logits,
            "predictions": predictions,
            "accuracy": accuracy,
        }
    return report

    


# Smoke check: compare accuracy on the deliberately misleading clips.
aggregation_smoke = evaluate_aggregation_methods(
    frame_model, video_batch, video_labels
)
print(
    "Aggregation smoke check:",
    {name: values["accuracy"] for name, values in aggregation_smoke.items()},
)


Aggregation smoke check: {'middle': 0.0, 'mean': 1.0, 'max': 0.0}


## Interpreting the Baseline

On the prepared clips, the middle frame is intentionally misleading, and its strong distractor also defeats max aggregation. Mean logits recover the repeated signal across the full clip. This toy result does **not** prove mean is always best; it shows why the temporal rule must match the event pattern.

In a real experiment, compare all methods on the same validation split and report Macro-F1, per-class recall, number of decoded frames, and runtime. Start with mean logits because it is simple and uses the whole sampled clip. Try max when a class may appear briefly, and keep middle-frame inference as a cheap latency baseline.

## Test Cases

Run this cell after completing Exercises 20-A through 20-D. It checks tensor shapes, dtypes, device preservation, temporal-axis reductions, inference mode, report schema, expected toy predictions, and error handling.

A correct implementation prints exactly `Day 20 tests passed`.

**Return structure — `run_day20_tests()`:** `None`. Success is communicated by completed assertions and the printed confirmation; a failed contract raises `AssertionError` or the expected input-validation exception.

In [15]:
def run_day20_tests():
    required_names = [
        "middle_frame_logits",
        "aggregate_frame_logits",
        "video_frame_logits",
        "evaluate_aggregation_methods",
    ]
    for name in required_names:
        assert name in globals(), "Missing function: " + name

    toy = torch.tensor(
        [[[1.0, 4.0], [2.0, 3.0], [8.0, 1.0]],
         [[5.0, 0.0], [6.0, 2.0], [7.0, 9.0]]],
        dtype=torch.float32,
    )
    middle = middle_frame_logits(toy)
    assert middle.shape == (2, 2)
    assert middle.dtype == toy.dtype and middle.device == toy.device
    assert torch.equal(middle, toy[:, 1, :])
    assert torch.allclose(aggregate_frame_logits(toy, "mean"), toy.mean(dim=1))
    assert torch.equal(aggregate_frame_logits(toy, "max"), toy.max(dim=1).values)

    model = MeanColorFrameClassifier()
    model.train()
    frame_logits = video_frame_logits(model, video_batch)
    assert frame_logits.shape == (B, T, 2)
    assert frame_logits.dtype == torch.float32
    assert frame_logits.device == video_batch.device
    assert model.training is False
    assert not frame_logits.requires_grad

    report = evaluate_aggregation_methods(model, video_batch, video_labels)
    assert set(report) == {"middle", "mean", "max"}
    for method in report:
        assert set(report[method]) == {"logits", "predictions", "accuracy"}
        assert report[method]["logits"].shape == (B, 2)
        assert report[method]["predictions"].shape == (B,)
        assert report[method]["predictions"].dtype == torch.long
        assert 0.0 <= report[method]["accuracy"] <= 1.0
    assert torch.equal(report["mean"]["predictions"], video_labels)
    assert report["mean"]["accuracy"] == 1.0
    assert report["middle"]["accuracy"] == 0.0
    assert report["max"]["accuracy"] == 0.0

    try:
        aggregate_frame_logits(toy, "median")
        raise AssertionError("Unsupported aggregation method should fail")
    except ValueError:
        pass
    try:
        middle_frame_logits(torch.empty(2, 0, 3))
        raise AssertionError("Empty time dimension should fail")
    except ValueError:
        pass
    try:
        video_frame_logits(model, torch.empty(2, 3, 8, 8))
        raise AssertionError("Four-dimensional video input should fail")
    except ValueError:
        pass
    try:
        evaluate_aggregation_methods(model, video_batch, video_labels.float())
        raise AssertionError("Non-long labels should fail")
    except ValueError:
        pass

    print("Day 20 tests passed")

run_day20_tests()


Day 20 tests passed


## Day 20 Checklist

- [ ] I can explain `[B, T, C, H, W]`, `[B*T, C, H, W]`, and `[B, T, K]`.
- [ ] I reshape frames without mixing different videos.
- [ ] I use `eval()` and `torch.no_grad()` for inference.
- [ ] I can implement middle, mean-logit, and max-logit aggregation.
- [ ] I understand when one misleading frame can hurt middle or max aggregation.
- [ ] I compare aggregation rules using the same frames, labels, and validation metric.
- [ ] I can trade off `B*T` frame inference cost against the cheaper middle-frame baseline.